In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go

In [2]:
class control_system:
    def __init__(self, dimentions, in_sys, w_scale = 0.1, keep_lenght = True):
        self.dims = dimentions
        self.in_sys = in_sys

        self.w = []

        for dc, dn in zip(self.dims, self.dims[1:]):
            w = np.random.randn(dc, dn, dn)
            if keep_lenght:
                w = w - np.transpose(w, (0, 2, 1))
            self.w.append(w*w_scale)

        self.state = []

        for d in self.dims:
            self.state.append(np.random.randn(d))

        def ds_dt(t, flat_state):
            # Reshape the flat state back to the original structure
            state = []
            idx = 0
            for d in self.dims:
                state.append(flat_state[idx:idx + d])
                idx += d

            ds = [None for _ in self.dims]
            ds[0] = self.in_sys(t, state[0])
            for i, (u, w, v) in enumerate(zip(state, self.w, state[1:])):
                ds[i + 1] = np.einsum("j, jki, k -> i", u, w, v, optimize=True)

            # Flatten the derivatives back into a 1D array
            return np.concatenate(ds)

        self.ds_dt = ds_dt

    def run(self, t_span=(0, 50), initials_in_sys=None, res=5000):
        if initials_in_sys is not None:
            self.state[0] = initials_in_sys

        # Flatten the initial state into a 1D array
        flat_state = np.concatenate([np.ravel(s) for s in self.state])

        t_eval = np.linspace(t_span[0], t_span[1], res)
        solution = solve_ivp(self.ds_dt, t_span, flat_state, t_eval=t_eval)

        # Reshape the solution back to the original structure
        reshaped_solution = []
        for i in range(len(solution.t)):
            idx = 0
            state_at_t = []
            for d in self.dims:
                state_at_t.append(solution.y[idx:idx + d, i])
                idx += d
            reshaped_solution.append(state_at_t)

        # Adjust reshaped_solution to ensure uniformity across layers
        reshaped_solution = [
            np.array([state_at_t[layer] for state_at_t in reshaped_solution])
            for layer in range(len(self.dims))
        ]

        return solution.t, reshaped_solution


def show_ev(evolution, time, title, random_projections = False):
    if not random_projections:
        x, y, z = [evolution[:, i] if i<evolution.shape[-1] else np.zeros_like(evolution[:, 0]) for i in range(3)]
    else:
        proj = evolution @ np.random.randn(evolution.shape[-1], 3)
        x, y, z = [proj[:, i] for i in range(3)]
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color=time, width=2), ))
    fig.update_layout(
        title = title,
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        template='plotly_dark',
    )

    fig.show()

In [3]:
sigma = 10.0
beta = 8.0 / 3.0
rho = 28.0

def lorenz_system(t, state):
    x, y, z = state
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return np.array([dxdt, dydt, dzdt])

In [26]:
sys = control_system((3, 4, 4, 3, 3), lorenz_system)#, w_scale=0.001, keep_lenght= False)

In [27]:
t, s = sys.run()

In [28]:
for n, l in enumerate(sys.dims):
    print(n)
    show_ev(s[n], t, title = f"layer №{n}, shape {l}", random_projections= False)

0


1


2


3


4
